In [6]:
# Load both raw files correctly

import pandas as pd

TRAIN_PATH = "../data/drugsComTrain_raw.csv"
TEST_PATH  = "../data/drugsComTest_raw.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print("TRAIN shape:", train.shape)
print("TEST  shape:", test.shape)

print("\nTRAIN columns:")
print(list(train.columns))

print("\nTEST columns:")
print(list(test.columns))

TRAIN shape: (161297, 7)
TEST  shape: (53766, 7)

TRAIN columns:
['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']

TEST columns:
['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']


In [7]:
# Keep only needed columns + show BEFORE/AFTER

# BEFORE
print("Before filtering:")
print("TRAIN columns count:", train.shape[1])
print("TEST  columns count:", test.shape[1])

print("\nTRAIN head(1) original:")
print(train.head(1))

print("\nTEST head(1) original:")
print(test.head(1))

# Keep only what we need (review + rating + condition for later EDA context)
keep_cols = ["review", "rating", "condition"]

train = train[keep_cols].copy()
test  = test[keep_cols].copy()

# AFTER
print("\nAfter filtering:")
print("TRAIN columns:", list(train.columns))
print("TEST  columns:", list(test.columns))

print("\nTRAIN head(1) filtered:")
print(train.head(1))

print("\nTEST head(1) filtered:")
print(test.head(1))

Before filtering:
TRAIN columns count: 7
TEST  columns count: 7

TRAIN head(1) original:
   uniqueID   drugName                     condition  \
0    206461  Valsartan  Left Ventricular Dysfunction   

                                              review  rating       date  \
0  "It has no side effect, I take it in combinati...       9  20-May-12   

   usefulCount  
0           27  

TEST head(1) original:
   uniqueID     drugName   condition  \
0    163740  Mirtazapine  Depression   

                                              review  rating       date  \
0  "I&#039;ve tried a few antidepressants over th...      10  28-Feb-12   

   usefulCount  
0           22  

After filtering:
TRAIN columns: ['review', 'rating', 'condition']
TEST  columns: ['review', 'rating', 'condition']

TRAIN head(1) filtered:
                                              review  rating  \
0  "It has no side effect, I take it in combinati...       9   

                      condition  
0  Left Ventricular

4 irrelevant columns early (uniqueID, drugName, date, usefulCount). kept only: review (text),rating (label source) and condition (optional contextual variable for later analysis)
This reduces noise and prevents accidental leakage (e.g., using usefulCount).

In [8]:
#Concatenate train + test
print("Before concat:")
print("Train shape:", train.shape)
print("Test  shape:", test.shape)

df = pd.concat([train, test], axis=0, ignore_index=True)

print("\nAfter concat:")
print("DF shape:", df.shape)

# Sanity check
expected_rows = train.shape[0] + test.shape[0]
print("\nExpected rows:", expected_rows)
print("Match check:", df.shape[0] == expected_rows)

Before concat:
Train shape: (161297, 3)
Test  shape: (53766, 3)

After concat:
DF shape: (215063, 3)

Expected rows: 215063
Match check: True


In [9]:
#Null check + drop missing review/rating

print("BEFORE dropping missing values")

print("\nMissing review count:", df["review"].isna().sum())
print("Missing rating count:", df["rating"].isna().sum())

# Show example rows with missing review (if any)
missing_review_rows = df[df["review"].isna()]
print("\nExample rows with missing review:")
print(missing_review_rows.head(3))

# Drop rows where review OR rating is missing
df = df.dropna(subset=["review", "rating"]).copy()

print("\nAFTER dropping missing values")

print("New shape:", df.shape)
print("Missing review count:", df["review"].isna().sum())
print("Missing rating count:", df["rating"].isna().sum())

BEFORE dropping missing values

Missing review count: 0
Missing rating count: 0

Example rows with missing review:
Empty DataFrame
Columns: [review, rating, condition]
Index: []

AFTER dropping missing values
New shape: (215063, 3)
Missing review count: 0
Missing rating count: 0


In [10]:
#Clean HTML entities (minimal repair)

import re
import html

# BEFORE: detect HTML entity patterns
html_pattern = r'&#|&amp;|&quot;'

contains_html = df["review"].str.contains(html_pattern, regex=True).sum()
print("Number of reviews containing HTML entities:", contains_html)

# Show 3 examples BEFORE cleaning
examples_before = df[df["review"].str.contains(html_pattern, regex=True)].head(3)
print("\nExamples BEFORE cleaning:")
for i, row in examples_before.iterrows():
    print("\n---")
    print(row["review"])

# Apply HTML unescape
df["review"] = df["review"].apply(lambda x: html.unescape(x))

# AFTER: check again
contains_html_after = df["review"].str.contains(html_pattern, regex=True).sum()
print("\nNumber of reviews containing HTML entities AFTER cleaning:", contains_html_after)

# Show the same 3 rows AFTER cleaning
print("\nExamples AFTER cleaning:")
for i in examples_before.index:
    print("\n---")
    print(df.loc[i, "review"])

Number of reviews containing HTML entities: 139250

Examples BEFORE cleaning:

---
"I used to take another oral contraceptive, which had 21 pill cycle, and was very happy- very light periods, max 5 days, no other side effects. But it contained hormone gestodene, which is not available in US, so I switched to Lybrel, because the ingredients are similar. When my other pills ended, I started Lybrel immediately, on my first day of period, as the instructions said. And the period lasted for two weeks. When taking the second pack- same two weeks. And now, with third pack things got even worse- my third period lasted for two weeks and now it&#039;s the end of the third week- I still have daily brown discharge.
The positive side is that I didn&#039;t have any other side effects. The idea of being period free was so tempting... Alas."

---
"This is my first time using any form of birth control. I&#039;m glad I went with the patch, I have been on it for 8 months. At first It decreased my libido 

### This Means
> 139,250 / 215,063 (~64.7%) of reviews contained HTML artifacts like &#039;.

> After html.unescape, 0 reviews still contain those patterns.

> This is a raw-text repair, not “full preprocessing,” and it prevents garbage tokens like 039, amp, etc.

In [11]:
# Create binary sentiment label + drop ambiguous ratings (5–6)

# Convert rating to numeric safely
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

print("Rating distribution BEFORE filtering (including potential NaN):")
print(df["rating"].value_counts(dropna=False).sort_index())

# Keep only valid rating range 1..10
df = df[df["rating"].between(1, 10)].copy()

# Drop ambiguous ratings 5 and 6
df = df[~df["rating"].isin([5, 6])].copy()

# Create sentiment
df["sentiment"] = (df["rating"] >= 7).astype(int)   # 7–10 -> 1, 1–4 -> 0

print("\nSentiment distribution AFTER filtering:")
print(df["sentiment"].value_counts())

print("\nSentiment distribution (percent):")
print(df["sentiment"].value_counts(normalize=True))

Rating distribution BEFORE filtering (including potential NaN):
rating
1     28918
2      9265
3      8718
4      6671
5     10723
6      8462
7     12547
8     25046
9     36708
10    68005
Name: count, dtype: int64

Sentiment distribution AFTER filtering:
sentiment
1    142306
0     53572
Name: count, dtype: int64

Sentiment distribution (percent):
sentiment
1    0.726503
0    0.273497
Name: proportion, dtype: float64


In [15]:
#before any capping
df_all = df.copy()

In [17]:
# Ensure sentiment exists & cap to EXACTLY 50,000 (stratified)

import pandas as pd
from sklearn.model_selection import train_test_split

TARGET = 50000

print("Columns BEFORE:", list(df.columns))
print("Rows BEFORE:", len(df))

# 1) Ensure sentiment exists (rebuild safely if missing)
if "sentiment" not in df.columns:
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df = df[df["rating"].between(1, 10)].copy()
    df = df[~df["rating"].isin([5, 6])].copy()
    df["sentiment"] = (df["rating"] >= 7).astype(int)

# 2) If df is larger than TARGET, downsample stratified to TARGET
if len(df) > TARGET:
    df, _ = train_test_split(
        df,
        train_size=TARGET,
        stratify=df["sentiment"],
        random_state=42
    )
    df = df.reset_index(drop=True)

# If df is smaller than TARGET (49,999), add rows safely without duplication issues
#    We pull from the full labeled pool (from your original merged df_all) if available.
#    If not available, we sample from df itself (will duplicate 1 row, acceptable for size constraint).
if len(df) < TARGET:
    need = TARGET - len(df)

    # Try to use a larger pool if it exists in your notebook
    pool = None
    if "df_all" in globals() and isinstance(df_all, pd.DataFrame) and "sentiment" in df_all.columns:
        pool = df_all
    else:
        pool = df

    # Match class proportions of current df
    frac_pos = (df["sentiment"] == 1).mean()
    n_pos = int(round(need * frac_pos))
    n_neg = need - n_pos

    extra_pos = pool[pool["sentiment"] == 1].sample(n=n_pos, replace=(len(pool[pool["sentiment"]==1]) < n_pos), random_state=42)
    extra_neg = pool[pool["sentiment"] == 0].sample(n=n_neg, replace=(len(pool[pool["sentiment"]==0]) < n_neg), random_state=42)

    extra = pd.concat([extra_pos, extra_neg], ignore_index=True)
    df = pd.concat([df, extra], ignore_index=True).reset_index(drop=True)

print("\nRows AFTER:", len(df))
print("\nSentiment counts AFTER:")
print(df["sentiment"].value_counts())
print("\nSentiment proportions AFTER:")
print(df["sentiment"].value_counts(normalize=True))
print("\nColumns AFTER:", list(df.columns))

Columns BEFORE: ['review', 'rating', 'condition', 'sentiment']
Rows BEFORE: 50000

Rows AFTER: 50000

Sentiment counts AFTER:
sentiment
1    36326
0    13674
Name: count, dtype: int64

Sentiment proportions AFTER:
sentiment
1    0.72652
0    0.27348
Name: proportion, dtype: float64

Columns AFTER: ['review', 'rating', 'condition', 'sentiment']


In [18]:
# Final Sanity Check 

print("==========SUMMARY ==========")

print("\nFinal number of rows:", len(df))
print("Final columns:", list(df.columns))

print("\nMissing values:")
print("Review missing:", df["review"].isna().sum())
print("Rating missing:", df["rating"].isna().sum())
print("Sentiment missing:", df["sentiment"].isna().sum())

print("\nSentiment counts:")
print(df["sentiment"].value_counts())

print("\nSentiment proportions:")
print(df["sentiment"].value_counts(normalize=True))

print("\nPreview (first 3 rows):")
print(df[["review", "rating", "sentiment", "condition"]].head(3))

print("=====================================")

==========SUMMARY ==========

Final number of rows: 50000
Final columns: ['review', 'rating', 'condition', 'sentiment']

Missing values:
Review missing: 0
Rating missing: 0
Sentiment missing: 0

Sentiment counts:
sentiment
1    36326
0    13674
Name: count, dtype: int64

Sentiment proportions:
sentiment
1    0.72652
0    0.27348
Name: proportion, dtype: float64

Preview (first 3 rows):
                                              review  rating  sentiment  \
0  "I have problem with frequent uriantion especi...       1          0   
1  "I actually thought this was the birth control...       1          0   
2  "I am really glad to see so many positive expe...       4          0   

                      condition  
0  Benign Prostatic Hyperplasia  
1                           NaN  
2                  Chronic Pain  


In [19]:
#Save processed dataset

import os

out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "reviews_processed.csv")

df.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("Saved shape:", df.shape)
print("Saved columns:", list(df.columns))

Saved to: ../data/processed\reviews_processed.csv
Saved shape: (50000, 4)
Saved columns: ['review', 'rating', 'condition', 'sentiment']
